# 05 · kallisto vs STAR
Same samples, trimming, annotation, DESeq2 code (`run_de`) and outlier exclusion; only the quantification differs.
Needs `results/star/counts_unstranded.csv.gz` from notebook 04. Kernel: **R** (`ir`).

In [ ]:
source('../R/helpers.R')
samples <- load_samples()
x <- load_txi(rownames(samples)); txi <- x$txi; genes <- x$genes
STAR_COUNTS <- file.path(ROOT, 'results/star/counts_unstranded.csv.gz')
star <- load_star_counts(STAR_COUNTS)[, rownames(samples)]
cat('kallisto', nrow(txi$counts), 'genes | STAR', nrow(star), 'genes | shared', length(intersect(rownames(txi$counts), rownames(star))), '\n')

## 1 · Counts per sample
Spearman correlation of gene counts (shared genes) between the two methods, and total counts.

In [ ]:
g <- intersect(rownames(txi$counts), rownames(star))
per_sample <- tibble(srr = rownames(samples), group = samples$group,
  spearman = map_dbl(rownames(samples), \(s) cor(txi$counts[g, s], star[g, s], method = 'spearman')),
  kallisto_M = colSums(txi$counts[g, ]) / 1e6, star_M = colSums(star[g, ]) / 1e6) %>% mutate(ratio_star_kallisto = round(star_M / kallisto_M, 2))
per_sample

In [ ]:
s <- rownames(samples)[1]
ggplot(tibble(k = txi$counts[g, s], s = star[g, s]), aes(log1p(k), log1p(s))) + geom_point(size = 0.3, alpha = 0.3) + geom_abline(color = 'red') +
  labs(title = paste('Gene counts,', s), x = 'log1p kallisto', y = 'log1p STAR')

## 2 · Differential expression with each method

In [ ]:
kal <- run_de(txi, samples, genes)
str <- run_de(star, samples, genes)
bind_rows(kallisto = bind_rows(lapply(kal, n_deg), .id = 'contrast'), STAR = bind_rows(lapply(str, n_deg), .id = 'contrast'), .id = 'method')

### Agreement per contrast
`jaccard` = shared / union of DEGs; `same_direction` = fraction of shared DEGs with the same sign; `lfc_spearman_*` = rank correlation of log2FC over all tested genes / genes significant in either.

In [ ]:
cmp <- bind_rows(lapply(names(kal), \(n) compare_contrast(kal[[n]], str[[n]]) %>% mutate(contrast = n))) %>% relocate(contrast)
cmp

In [ ]:
plots <- lapply(names(kal), \(n) lfc_scatter(kal[[n]], str[[n]], n))
for (p in plots) print(p)

## 3 · Genes called by only one method
Are the disagreements low-expression genes? (median mean-count per class)

In [ ]:
only <- function(a, b, alpha = 0.05) inner_join(a, b, by = 'gene_id', suffix = c('_k', '_s')) %>% filter(!is.na(padj_k), !is.na(padj_s)) %>%
  mutate(call = case_when(padj_k < alpha & padj_s < alpha ~ 'both', padj_k < alpha ~ 'kallisto only', padj_s < alpha ~ 'STAR only', TRUE ~ 'neither'))
bind_rows(lapply(names(kal), \(cn) only(kal[[cn]], str[[cn]]) %>% filter(call != 'neither') %>% group_by(call) %>%
  summarise(n_genes = n(), median_baseMean = round(median((baseMean_k + baseMean_s) / 2)), .groups = 'drop') %>% mutate(contrast = cn))) %>% relocate(contrast)

## 4 · Genes highlighted in the paper

In [ ]:
key <- list(aging_21v3 = c('C4b', 'Trem2', 'Gpr17', 'Gfap', 'Ctss', 'Tyrobp'), aDR = c('Cry1', 'Usp2', 'Ciart', 'Cirbp', 'Dbp'))
bind_rows(lapply(names(key), \(n) {
  f <- \(r, m) r[[n]] %>% filter(symbol %in% key[[n]]) %>% transmute(contrast = n, symbol, !!paste0('lfc_', m) := round(log2FoldChange, 2), !!paste0('padj_', m) := signif(padj, 2))
  inner_join(f(kal, 'kallisto'), f(str, 'STAR'), by = c('contrast', 'symbol'))
}))

## 5 · Save

In [ ]:
dir.create(file.path(ROOT, 'results/compare'), showWarnings = FALSE)
write_csv(cmp, file.path(ROOT, 'results/compare/de_agreement.csv'))
write_csv(per_sample, file.path(ROOT, 'results/compare/counts_per_sample.csv'))
imap(str, \(r, n) write_csv(r, file.path(ROOT, 'results/compare', paste0('star_', n, '.csv')))) %>% invisible()
list.files(file.path(ROOT, 'results/compare'))